# BenignIDS — Orchestrator (nbclient)


# BenignIDS — Orchestrator (nbclient, auto-discovery)

**Purpose**
- Auto-discovers all notebooks named like `NN-*.ipynb` (e.g., `01-Setup_Preflight.ipynb` … `10-Deployment_Drift.ipynb`).
- Executes them in numeric order under isolated kernels with `nbclient`.
- Persists executed copies to `./executed_from_driver/`.

**Controls**
- `STOP_ON_ERROR` **must** be `True` to stop on the first failure, or `False` to continue.
- `INCLUDE_PLACEHOLDERS` **should** be `False` to skip placeholder notebooks whose first cell contains the word `Placeholder`.
- `TIMEOUT_SECONDS` **should** be set per your environment; default `900`.
- `KERNEL_NAME` **may** be adjusted if you use a non-default kernel.


In [2]:
# ======================================================
# Orchestrator — nbclient (isolated kernels, auto-discovery)
#   • Discovers NN-*.ipynb (excluding this orchestrator)
#   • Executes in numeric order, saves to ./executed_from_driver/
#   • Normalises cell IDs to silence nbformat warnings
# ======================================================
print(">>> Orchestrator — nbclient (isolated kernels, auto-discovery)")

from nbclient import NotebookClient
import nbformat, re
from pathlib import Path
from datetime import datetime, timezone
import uuid

# ---- Controls
STOP_ON_ERROR = False         # set to False to continue after errors
INCLUDE_PLACEHOLDERS = False  # skip notebooks whose first cell says 'Placeholder'
TIMEOUT_SECONDS = 900
KERNEL_NAME = "python3"
ALLOW_ERRORS = True           # allow cells with errors to continue

base = Path(".")
out_dir = base / "executed_from_driver"
out_dir.mkdir(exist_ok=True)

def _normalize_ids(nb):
    changed = 0
    for cell in nb.get("cells", []):
        if "id" not in cell:
            cell["id"] = str(uuid.uuid4())
            changed += 1
    return changed

# ---- Discover notebooks: NN-*.ipynb (exclude orchestrators)
rx = re.compile(r"^(\d{2})-.*\.ipynb$")
candidates = []
for p in sorted(base.glob("*.ipynb")):
    name = p.name
    if "Orchestrator" in name:
        continue
    m = rx.match(name)
    if m:
        order = int(m.group(1))
        candidates.append((order, p))

ORDER = [str(p) for _, p in sorted(candidates, key=lambda x: x[0])]
print(f"[driver] Discovered {len(ORDER)} notebooks:")
for n in ORDER:
    print("  -", n)

summary = []
start_all = datetime.now(timezone.utc)
for nb_path in ORDER:
    src = base / nb_path
    try:
        nb = nbformat.read(src, as_version=4)
        # Optionally skip placeholders
        if not INCLUDE_PLACEHOLDERS:
            try:
                first = nb["cells"][0]
                if first.get("cell_type") == "markdown" and any("Placeholder" in s for s in first.get("source", [])):
                    print(f"[driver][skip] Placeholder: {src}")
                    summary.append({"notebook": nb_path, "status": "skipped_placeholder"})
                    continue
            except Exception:
                pass
        # Normalise IDs
        changed = _normalize_ids(nb)
        if changed:
            print(f"[driver] Normalised {changed} missing cell IDs in {src.name}")
        # Execute
        print(f"[driver] Executing: {src}")
        client = NotebookClient(nb, timeout=TIMEOUT_SECONDS, kernel_name=KERNEL_NAME)
        client.execute()
        dst = out_dir / src.name
        nbformat.write(nb, dst)
        print(f"[driver] Wrote executed notebook: {dst}")
        summary.append({"notebook": nb_path, "status": "ok", "path": str(dst)})
    except Exception as e:
        print(f"[driver][ERROR] Failure in {nb_path}: {e}")
        summary.append({"notebook": nb_path, "status": "error", "error": str(e)})
        if STOP_ON_ERROR:
            break

elapsed = (datetime.now(timezone.utc) - start_all).total_seconds()
print(f"[driver] Orchestrator finished in {elapsed:.1f}s")
print("[driver] Summary:")
for row in summary:
    print("  -", row)


>>> Orchestrator — nbclient (isolated kernels, auto-discovery)
[driver] Discovered 9 notebooks:
  - 01-02-Setup_and_Feature_Engineering.ipynb
  - 03-Feature_Selection.ipynb
  - 04-Baseline_and_BO.ipynb
  - 05-HPO.ipynb
  - 06-Ensembles.ipynb
  - 07-CNN.ipynb
  - 08-Reporting.ipynb
  - 09-Champion_AutoWire.ipynb
  - 10-Deployment_Drift.ipynb
[driver] Executing: 01-02-Setup_and_Feature_Engineering.ipynb
[driver] Wrote executed notebook: executed_from_driver/01-02-Setup_and_Feature_Engineering.ipynb
[driver] Executing: 03-Feature_Selection.ipynb
[driver] Wrote executed notebook: executed_from_driver/03-Feature_Selection.ipynb
[driver] Executing: 04-Baseline_and_BO.ipynb
[driver][ERROR] Failure in 04-Baseline_and_BO.ipynb: An error occurred while executing the following cell:
------------------
# ======================================================
# Section 4.3 — BO Trials & Evaluation (payload-only) [TIME-BOXED PATCH]
#   • gp_minimize with a strict time budget + checkpoint every trial

/opt/miniconda3/envs/py312/lib/python3.12/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


[driver] Wrote executed notebook: executed_from_driver/08-Reporting.ipynb
[driver] Executing: 09-Champion_AutoWire.ipynb
[driver][ERROR] Failure in 09-Champion_AutoWire.ipynb: An error occurred while executing the following cell:
------------------
# =====================================================
# Section 9 — Section
# =====================================================
print(">>> Section 9: start")
reports_dir = Path(OUT_ROOT) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
if champion_valid:
    with (reports_dir / 'champion.json').open('w') as f:
        json.dump(champion_obj, f, indent=2)
    print('[ok] champion.json saved →', reports_dir / 'champion.json')
else:
    precheck_missing = tuple(globals().get('PRECHECK_MISSING', ()))
    required_sections = ['Section 4 — Train LightGBM (baseline or BO)', 'Section 5.* — HPO variants (if used in selection)', 'Section 6.1 — Baseline Model & Metrics (if metrics aggregation happens there)']
    payload_needed = 'paylo